# Mycelium UV-response analysis

Reproducible analysis of extracellular voltage recordings from mycelium samples exposed to repeated UV illumination, with and without static loading.

### Datasets
- **No weight:** `mycelium_ch1.npy`
- **Weight 1:** `mycelium_ch1_weight.npy`
- **Weight 2:** `mycelium_ch1_weight2.npy`
- **Agar control:** `agar_control_ch3.npy`

The notebook follows the student's original filtering approach and then performs the additional cycle-aligned and kinetic analysis developed in this study.

## 1. Imports and configuration

Sampling is 100 Hz. The UV protocol is aligned at 90 s, with 60 s UV ON followed by a nominal 180 s recovery period.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

FS = 100.0
UV_START = 90.0
CYCLE_PERIOD = 240.0
UV_ON = 60.0
PRE = 30.0
POST = 180.0

ROOT = Path.cwd()
DATA = ROOT / 'data'
FIG = ROOT / 'figures'
RESULTS = ROOT / 'results'
FIG.mkdir(exist_ok=True)
RESULTS.mkdir(exist_ok=True)

FILES = {
    'No weight': DATA / 'mycelium_ch1.npy',
    'Weight 1': DATA / 'mycelium_ch1_weight.npy',
    'Weight 2': DATA / 'mycelium_ch1_weight2.npy',
    'Agar control': DATA / 'agar_control_ch3.npy',
}

print('Working directory:', ROOT)
print('Data directory:', DATA)

## 2. Load and inspect the raw recordings

Voltage is converted from volts to millivolts. The student's Savitzky–Golay filtering is retained: 101 samples and polynomial order 3.

In [ ]:
def load_and_filter(path):
    x = np.load(path).astype(float) * 1000.0
    y = savgol_filter(x, 101, 3)
    return x, y

raw = {}
filtered = {}

for label, path in FILES.items():
    if path.exists():
        raw[label], filtered[label] = load_and_filter(path)
        print(f'{label:12s}: {len(raw[label]) / FS:.1f} s, {len(raw[label])} samples')
    else:
        print('Missing:', path)

In [ ]:
# Full recordings
for label, y in filtered.items():
    t = np.arange(len(y)) / FS
    fig = plt.figure(figsize=(11, 4))
    ax = fig.add_subplot(111)
    ax.plot(t / 60, y, linewidth=0.9)
    ax.set_xlabel('Time (min)')
    ax.set_ylabel('Voltage (mV)')
    ax.set_title(label)
    ax.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()

## 3. Cycle alignment and local baseline correction

Each cycle is aligned to UV ON. The preceding 30 s is used as a local baseline, and its median is subtracted from that cycle. This removes DC offsets and focuses the analysis on the UV-associated change.

In [ ]:
def aligned_cycles(path, ncycles=30, post_window=180):
    _, y = load_and_filter(path)
    duration = len(y) / FS
    traces, ids = [], []

    for c in range(ncycles):
        t0 = UV_START + c * CYCLE_PERIOD
        ia = int((t0 - PRE) * FS)
        ib = int((t0 + post_window) * FS)
        if ia < 0 or ib > len(y):
            continue
        tr = y[ia:ib].copy()
        baseline = np.median(y[ia:int(t0 * FS)])
        tr -= baseline
        traces.append(tr)
        ids.append(c + 1)

    if not traces:
        return None, None, []

    arr = np.vstack(traces)
    t = np.arange(arr.shape[1]) / FS - PRE
    return t, arr, ids

myel = {}
for label in ['No weight', 'Weight 1', 'Weight 2']:
    myel[label] = aligned_cycles(FILES[label], 30, POST)

# The agar recording is short, so only the available cycles are used.
agar = aligned_cycles(FILES['Agar control'], 2, 80)

for label, (t, arr, ids) in myel.items():
    print(label, ':', arr.shape, 'cycles')
print('Agar control:', agar[1].shape if agar[1] is not None else None)

## 4. Cycle-aligned median response

The median is used as a robust phase average. The shaded region is the 25th–75th percentile range across cycles.

In [ ]:
fig = plt.figure(figsize=(11, 5.5))
ax = fig.add_subplot(111)

for label, (t, arr, ids) in myel.items():
    med = np.median(arr, axis=0)
    q25 = np.percentile(arr, 25, axis=0)
    q75 = np.percentile(arr, 75, axis=0)
    ax.fill_between(t, q25, q75, alpha=0.12)
    ax.plot(t, med, linewidth=2, label=label)

if agar[0] is not None:
    t_agar, arr_agar, _ = agar
    ax.plot(t_agar, np.median(arr_agar, axis=0), linewidth=1.7, label='Agar control')

ax.axvspan(0, 60, alpha=0.10, label='UV ON')
ax.axhline(0, linewidth=0.8)
ax.set_xlim(-30, 180)
ax.set_xlabel('Time relative to UV ON (s)')
ax.set_ylabel('ΔV relative to pre-UV baseline (mV)')
ax.set_title('Cycle-aligned extracellular voltage response')
ax.grid(alpha=0.2)
ax.legend()
plt.tight_layout()
plt.show()

## 5. Model-independent kinetic descriptors

For each cycle we calculate:

- $A_{max}$: maximum response during UV ON;
- early minimum: minimum during the first 20 s;
- sustained response: median response during 50–60 s;
- refined $t_{50}$: time after the early minimum to reach 50% of the rise toward the sustained response;
- refined $t_{90}$: corresponding 90% time;
- recovery half-time: time after UV OFF to reach half of the voltage immediately before UV OFF;
- UV AUC: integral of the baseline-subtracted response during the 60 s UV exposure.

The refined $t_{50}$ is deliberately model-independent so that biphasic responses are not forced into a single-exponential model.

In [ ]:
def first_crossing(t, y, target, direction='up'):
    if direction == 'up':
        idx = np.where(y >= target)[0]
    else:
        idx = np.where(y <= target)[0]
    if len(idx) == 0:
        return np.nan
    i = idx[0]
    if i == 0 or y[i] == y[i-1]:
        return float(t[i])
    return float(t[i-1] + (target-y[i-1])*(t[i]-t[i-1])/(y[i]-y[i-1]))

def cycle_metrics(t, arr, ids, condition):
    rows = []
    for j, cycle in enumerate(ids):
        tr = arr[j]
        early_mask = (t >= 0) & (t < 20)
        sustained_mask = (t >= 50) & (t < 60)
        uv_mask = (t >= 0) & (t < 60)
        recovery_mask = (t >= 60) & (t < 180)

        early_indices = np.where(early_mask)[0]
        i_min = early_indices[np.argmin(tr[early_indices])]
        t_min = t[i_min]
        v_min = tr[i_min]

        v_sustained = np.median(tr[sustained_mask])
        target50 = v_min + 0.5*(v_sustained-v_min)
        target90 = v_min + 0.9*(v_sustained-v_min)

        post_min = (t >= max(0, t_min)) & (t < 60)
        t_uv = t[post_min]
        y_uv = tr[post_min]
        t50 = first_crossing(t_uv, y_uv, target50, 'up')
        t90 = first_crossing(t_uv, y_uv, target90, 'up')

        amax = np.max(tr[uv_mask])
        t_peak = t[uv_mask][np.argmax(tr[uv_mask])]

        v_off = np.median(tr[(t >= 59.5) & (t < 60)])
        t_rec = t[recovery_mask] - 60
        y_rec = tr[recovery_mask]
        target_rec = 0.5*v_off
        idx = np.where(y_rec <= target_rec)[0] if v_off >= 0 else np.where(y_rec >= target_rec)[0]
        recovery_half = np.nan if len(idx) == 0 else float(t_rec[idx[0]])

        auc = np.trapezoid(tr[uv_mask], t[uv_mask])

        rows.append({
            'Condition': condition,
            'Cycle': cycle,
            'Amax_mV': amax,
            't_peak_s': t_peak,
            't_min_s': t_min,
            'early_min_mV': v_min,
            'sustained_mV': v_sustained,
            't50_refined_s': t50,
            't90_refined_s': t90,
            'recovery_half_s': recovery_half,
            'UV_AUC_mV_s': auc,
        })
    return pd.DataFrame(rows)

tables = []
for label, (t, arr, ids) in myel.items():
    tables.append(cycle_metrics(t, arr, ids, label))

kin = pd.concat(tables, ignore_index=True)
kin.head()

## 6. Cycle-by-cycle kinetics

These plots show whether the response remains stable, adapts, or drifts over repeated UV stimulation.

In [ ]:
plot_specs = [
    ('Amax_mV', 'Amax (mV)', 'Cycle-to-cycle UV response amplitude'),
    ('t50_refined_s', 't50 (s)', 'Cycle-to-cycle response speed'),
    ('recovery_half_s', 'Recovery half-time (s)', 'Cycle-to-cycle recovery kinetics'),
    ('UV_AUC_mV_s', 'UV AUC (mV·s)', 'Cycle-to-cycle integrated UV response'),
]

for col, ylabel, title in plot_specs:
    fig = plt.figure(figsize=(10, 4.5))
    ax = fig.add_subplot(111)
    for label in ['No weight', 'Weight 1', 'Weight 2']:
        d = kin[kin.Condition == label]
        ax.plot(d.Cycle, d[col], 'o-', ms=3, linewidth=1.1, label=label)
    ax.set_xlabel('UV cycle')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.2)
    ax.legend()
    plt.tight_layout()
    plt.show()

## 7. Cycle-by-cycle response matrices

Each row is one UV exposure and each column is time relative to UV activation.

In [ ]:
for label, (t, arr, ids) in myel.items():
    fig = plt.figure(figsize=(10, 5.6))
    ax = fig.add_subplot(111)
    im = ax.imshow(
        arr, aspect='auto', origin='upper',
        extent=[t[0], t[-1], len(ids)+0.5, 0.5],
        interpolation='nearest'
    )
    ax.axvspan(0, 60, alpha=0.10)
    ax.set_xlabel('Time relative to UV ON (s)')
    ax.set_ylabel('UV cycle')
    ax.set_title(f'{label}: cycle-by-cycle response')
    cb = fig.colorbar(im, ax=ax)
    cb.set_label('ΔV (mV)')
    plt.tight_layout()
    plt.show()

## 8. Summary statistics

These are descriptive statistics across repeated cycles within each experimental recording. They are **not** independent biological replicates.

In [ ]:
summary = kin.groupby('Condition').agg(
    n=('Cycle', 'count'),
    Amax_mean=('Amax_mV', 'mean'),
    Amax_median=('Amax_mV', 'median'),
    Amax_sd=('Amax_mV', 'std'),
    t50_mean=('t50_refined_s', 'mean'),
    t50_median=('t50_refined_s', 'median'),
    t50_sd=('t50_refined_s', 'std'),
    recovery_half_mean=('recovery_half_s', 'mean'),
    recovery_half_median=('recovery_half_s', 'median'),
    recovery_half_sd=('recovery_half_s', 'std'),
    AUC_mean=('UV_AUC_mV_s', 'mean'),
    AUC_median=('UV_AUC_mV_s', 'median'),
).reset_index()

display(summary.round(4))

## 9. Save processed results

The cycle-level table and summary can be exported directly for subsequent statistical analysis.

In [ ]:
kin_path = RESULTS / 'cycle_kinetics_refined.csv'
summary_path = RESULTS / 'kinetics_summary_refined.csv'

kin.to_csv(kin_path, index=False)
summary.to_csv(summary_path, index=False)

print('Saved:', kin_path)
print('Saved:', summary_path)

## 10. Interpretation notes

The current data are best regarded as **pilot observations**. There is one unloaded recording and two recordings with the same nominal static weight. The 30 UV cycles are repeated measurements within an experimental run and should not be treated as 30 independent biological replicates.

The current observations are:

1. UV produces a reproducible time-dependent extracellular voltage response in the mycelium recordings.
2. The unloaded and first loaded recordings show an initial negative component followed by a slower positive response.
3. The second loaded recording shows a strong, predominantly positive and comparatively rapid response.
4. The second loaded recording also shows a progressive reduction in response amplitude over repeated UV cycles.
5. The agar control does not reproduce the characteristic positive response seen in the mycelium recordings, although the control recording is much shorter.
6. The two nominally identical loaded experiments differ substantially, so the present data do not establish a causal effect of static loading.

For a definitive loading study, independent mycelium samples—not individual UV cycles—should be used as the independent experimental units.